# Reading DNA with a convolutional neural network: finding promoters and recovering their motifs

**Hafsa Fathima Khajamainudeen** · M.Sc. Bioinformatics & Data Science

A promoter is the stretch of DNA that tells the cell where a gene starts. This notebook trains a
small 1D convolutional neural network to read raw DNA sequence (just the letters A, C, G, T) and
decide whether a sequence is a promoter, then opens the trained network up to show that its filters
have **learned real sequence motifs on their own** — the same short patterns biologists use to
recognise promoters by eye.

The point of the project is not only "can a CNN classify DNA" (it can), but that a convolutional
filter over DNA *is* a motif detector, and after training we can read the motif back out of the
weights. That is the idea behind DeepBind and the whole family of genomic deep-learning models.

**Data.** The [`genomic-benchmarks`](https://github.com/ML-Bioinfo-CEITEC/genomic_benchmarks)
`human_nontata_promoters` set: real human promoter sequences (non-TATA promoters) versus length-matched
non-promoter sequences. It downloads itself. If the download is unavailable, the notebook falls back to a
synthetic dataset with a planted motif, so every cell below still runs and the motif-recovery step still
has something to recover.

Run top to bottom on a GPU runtime (Runtime → Change runtime type → T4 GPU). ~2–4 minutes.


In [ ]:
# Install dependencies (Colab)
!pip -q install logomaker datasets 2>/dev/null
print("done")

In [ ]:
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, accuracy_score, roc_auc_score
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
BASES = "ACGT"; B2I = {b:i for i,b in enumerate(BASES)}

## 1. Load the data

Each sequence is a real human genomic window (~251bp): positives are non-TATA promoters, negatives are
length-matched non-promoter DNA, from the `human_nontata_promoters` benchmark. We try two real sources
in order — a Hugging Face mirror (carries the actual sequences, downloads reliably on Colab) and the
original Zenodo archive — and print the full error for any that fails. Only if both are unreachable does
the notebook fall back to synthetic data (sequences with a planted `TATAAA`) so the pipeline and the
motif-recovery step still run end to end.

We deliberately do **not** use the `genomic-benchmarks` package loader: it depends on the deprecated
`torchtext`, which is broken on current Colab PyTorch.

In [ ]:
import urllib.request, zipfile, os, glob, tempfile, traceback

def load_hf():
    from datasets import load_dataset
    rid = "katarinagresova/Genomic_Benchmarks_human_nontata_promoters"
    ds = load_dataset(rid)
    def pick(split):
        d = ds[split]; cols = d.column_names; row0 = d[0]
        seqcol = next(c for c in cols if isinstance(row0[c], str)
                      and len(row0[c]) > 20 and set(row0[c].upper()[:40]) <= set("ACGTN"))
        labcol = next(c for c in cols if c != seqcol)
        S = [str(x).upper() for x in d[seqcol]]
        def tol(x):
            if isinstance(x, str): return 1 if ("pos" in x.lower() or x.strip() == "1") else 0
            return int(x)
        return S, np.array([tol(x) for x in d[labcol]])
    trs, try_ = pick("train")
    tes, tey = pick("test" if "test" in ds else "validation")
    return trs, try_, tes, tey, f"real: HF {rid}"

def _read_gb_folder(root):
    # genomic-benchmarks on-disk format: <root>/.../<split>/<label_folder>/<id>.txt, one sequence per file
    def read_split(split):
        sdir = None
        for d in glob.glob(os.path.join(root, "**", split), recursive=True):
            if os.path.isdir(d) and any(os.path.isdir(os.path.join(d, x)) for x in os.listdir(d)):
                sdir = d; break
        if sdir is None: raise FileNotFoundError(f"no '{split}' folder in archive")
        label_folders = sorted(x for x in os.listdir(sdir) if os.path.isdir(os.path.join(sdir, x)))
        S, Y = [], []
        for x in label_folders:
            lab = 1 if ("pos" in x.lower() or x == "1") else 0
            for fp in glob.glob(os.path.join(sdir, x, "*.txt")):
                with open(fp) as fh: s = fh.read().strip().upper()
                if s: S.append(s); Y.append(lab)
        return S, np.array(Y)
    return read_split("train"), read_split("test")

def load_zenodo():
    url = "https://zenodo.org/records/16605299/files/human_nontata_promoters_v0.zip?download=1"
    tmp = tempfile.mkdtemp(); zp = os.path.join(tmp, "hnp.zip")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=120) as r, open(zp, "wb") as fh:
        fh.write(r.read())
    with zipfile.ZipFile(zp) as z: z.extractall(tmp)
    (trs, try_), (tes, tey) = _read_gb_folder(tmp)
    if len(trs) == 0 or len(tes) == 0: raise RuntimeError("archive parsed to 0 sequences")
    return trs, try_, tes, tey, "real: human_nontata_promoters (Zenodo 16605299)"

def load_synth():
    L, MOTIF = 251, "TATAAA"
    def rseq(n): return "".join(np.random.choice(list(BASES), n))
    def make(npos, nneg):
        S, Y = [], []
        for _ in range(npos):
            a = list(rseq(L)); p = np.random.randint(10, L-len(MOTIF)-10)
            a[p:p+len(MOTIF)] = list(MOTIF); S.append("".join(a)); Y.append(1)
        for _ in range(nneg): S.append(rseq(L)); Y.append(0)
        i = np.random.permutation(len(S)); return [S[j] for j in i], np.array(Y)[i]
    trs, try_ = make(4000, 4000); tes, tey = make(1000, 1000)
    return trs, try_, tes, tey, "synthetic (planted TATAAA motif) — real download unavailable"

train_seqs = None
for loader in (load_hf, load_zenodo):
    try:
        train_seqs, train_y, test_seqs, test_y, SOURCE = loader()
        break
    except Exception:
        print(f"[{loader.__name__} FAILED]\n" + traceback.format_exc() + "-"*60)
if train_seqs is None:
    train_seqs, train_y, test_seqs, test_y, SOURCE = load_synth()

banner = "REAL HUMAN PROMOTER DATA" if SOURCE.startswith("real") else "SYNTHETIC FALLBACK (real data unavailable)"
print("\n" + "="*46 + f"\n  {banner}\n  source: {SOURCE}\n" + "="*46)
print("train", len(train_seqs), "| test", len(test_seqs))
print("class balance (train):", np.bincount(train_y))
print("example:", train_seqs[0][:60], "...  label", train_y[0])

## 2. One-hot encode the sequences

Each base becomes a 4-channel column: A=[1,0,0,0], C=[0,1,0,0], and so on. A sequence of length *L*
becomes a 4×*L* image, which a 1D convolution slides along exactly like a motif scanner. Any letter
other than A/C/G/T (an N, say) is left as all-zeros.

In [ ]:
# pad/truncate to a common length so we can batch
LEN = int(np.median([len(s) for s in train_seqs]))
def one_hot(seqs, L=LEN):
    X = np.zeros((len(seqs), 4, L), dtype=np.float32)
    for i, s in enumerate(seqs):
        for j, b in enumerate(s[:L]):
            k = B2I.get(b, None)
            if k is not None: X[i, k, j] = 1.0
    return X
Xtr = one_hot(train_seqs); Xte = one_hot(test_seqs)
print("encoded:", Xtr.shape, Xte.shape, "(sequences, 4 bases, length)")

## 3. The model

A single convolutional layer of motif detectors, a global max-pool (*does this motif appear
anywhere in the sequence?*), and a linear layer that weighs the detectors into a promoter score.
Deliberately small and interpretable — each of the 32 filters is an 11bp motif we can read out later.

In [ ]:
class DNACNN(nn.Module):
    def __init__(self, n_filters=32, k=11):
        super().__init__()
        self.conv = nn.Conv1d(4, n_filters, kernel_size=k)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.drop = nn.Dropout(0.3)
        self.fc   = nn.Linear(n_filters, 1)
        self.k, self.n_filters = k, n_filters
    def forward(self, x):
        h = self.relu(self.conv(x))
        h = self.pool(h).squeeze(-1)
        return self.fc(self.drop(h)).squeeze(-1)

model = DNACNN().to(DEVICE)
print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))

## 4. Train

Standard binary cross-entropy, Adam, with a 10% validation split to watch for overfitting.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
# validation split
n = len(Xtr); idx = np.random.permutation(n); cut = int(0.9*n)
tr_i, va_i = idx[:cut], idx[cut:]
Xtr_t = torch.tensor(Xtr); ytr_t = torch.tensor(train_y, dtype=torch.float32)
train_dl = DataLoader(TensorDataset(Xtr_t[tr_i], ytr_t[tr_i]), batch_size=128, shuffle=True)
Xva = Xtr_t[va_i].to(DEVICE); yva = train_y[va_i]
Xte_t = torch.tensor(Xte).to(DEVICE)

opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
lossf = nn.BCEWithLogitsLoss()
EPOCHS = 12
for ep in range(EPOCHS):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad(); lossf(model(xb), yb).backward(); opt.step()
    model.eval()
    with torch.no_grad():
        pva = torch.sigmoid(model(Xva)).cpu().numpy()
    print(f"epoch {ep+1:2d}  val AUC {roc_auc_score(yva, pva):.3f}  val acc {accuracy_score(yva, pva>0.5):.3f}")

## 5. Evaluate on the held-out test set

In [ ]:
model.eval()
with torch.no_grad():
    p_test = torch.sigmoid(model(Xte_t)).cpu().numpy()
test_auc = roc_auc_score(test_y, p_test)
test_acc = accuracy_score(test_y, p_test > 0.5)
print(f"TEST  ROC-AUC {test_auc:.3f}   accuracy {test_acc:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 4.2))
fpr, tpr, _ = roc_curve(test_y, p_test); rocauc = auc(fpr, tpr)
ax[0].plot(fpr, tpr, color="#2f6fd0", lw=2, label=f"CNN (AUC={rocauc:.3f})")
ax[0].plot([0,1],[0,1],"--",color="#999")
ax[0].set_xlabel("False positive rate"); ax[0].set_ylabel("True positive rate")
ax[0].set_title("ROC"); ax[0].legend(loc="lower right")
pr, rc, _ = precision_recall_curve(test_y, p_test); prauc = auc(rc, pr)
ax[1].plot(rc, pr, color="#2a9d6a", lw=2, label=f"CNN (AP={prauc:.3f})")
ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision")
ax[1].set_title("Precision-Recall"); ax[1].legend(loc="lower left")
plt.tight_layout(); plt.savefig("performance.png", dpi=130, bbox_inches="tight"); plt.show()

## 6. Read the motifs back out of the network

This is the interesting part. For each convolutional filter, we find the short DNA windows across the
real promoters that activate it most strongly, and stack them into a position weight matrix (PWM).
If the network has learned anything biological, these PWMs will look like real promoter motifs
(on the synthetic data, they should reproduce the planted `TATAAA`).

We rank filters by how much the final layer relies on them, and draw the top ones as sequence logos
— taller letters carry more information.

In [ ]:
# activation of each filter at each position, on positive (promoter) sequences
pos_mask = train_y == 1
pos_X = Xtr[pos_mask]
with torch.no_grad():
    acts = model.relu(model.conv(torch.tensor(pos_X).to(DEVICE))).cpu().numpy()  # (Npos, nf, W)
k = model.k
fc_w = model.fc.weight.detach().cpu().numpy().ravel()
top_filters = np.argsort(-np.abs(fc_w))[:4]   # 4 most influential filters

def filter_pwm(f, pct=99.7):
    A = acts[:, f, :]
    thr = np.percentile(A, pct)
    pwm = np.zeros((4, k)); cnt = 0
    hits = np.argwhere(A >= thr)
    for si, p in hits:
        win = pos_X[si, :, p:p+k]
        if win.shape[1] == k:
            pwm += win; cnt += 1
    if cnt: pwm /= cnt
    return pwm, cnt

def consensus(pwm): return "".join(BASES[np.argmax(pwm[:, j])] for j in range(pwm.shape[1]))

In [ ]:
# draw sequence logos (logomaker if present, else matplotlib information-content letters)
def ic_logo_mpl(pwm, ax, title):
    kk = pwm.shape[1]; colors={"A":"#2ca02c","C":"#1f77b4","G":"#ff7f0e","T":"#d62728"}
    for j in range(kk):
        col = pwm[:, j] + 1e-9; col = col/col.sum()
        ic = 2 + np.sum(col*np.log2(col))
        y0 = 0
        for idx in np.argsort(col):
            h = col[idx]*ic
            ax.text(j+0.5, y0+h/2, BASES[idx], ha="center", va="center",
                    fontsize=5+30*h, fontweight="bold", color=colors[BASES[idx]], family="monospace")
            y0 += h
    ax.set_xlim(0,kk); ax.set_ylim(0,2); ax.set_xticks(np.arange(kk)+0.5)
    ax.set_xticklabels(range(1,kk+1), fontsize=8); ax.set_ylabel("bits"); ax.set_title(title, fontsize=10)

try:
    import logomaker, pandas as pd
    HAVE_LM = True
except Exception:
    HAVE_LM = False

fig, axes = plt.subplots(len(top_filters), 1, figsize=(6, 2.3*len(top_filters)))
pwms = {}
for ax, f in zip(np.atleast_1d(axes), top_filters):
    pwm, cnt = filter_pwm(f); pwms[int(f)] = pwm
    cons = consensus(pwm)
    if HAVE_LM:
        df = pd.DataFrame(pwm.T, columns=list(BASES))
        info = df.values + 1e-9; info = info/info.sum(1, keepdims=True)
        ic = (2 + (info*np.log2(info)).sum(1, keepdims=True))
        logomaker.Logo(pd.DataFrame(info*ic, columns=list(BASES)), ax=ax)
        ax.set_ylim(0,2); ax.set_ylabel("bits")
        ax.set_title(f"filter {f}  (consensus {cons}, weight {fc_w[f]:+.2f})", fontsize=10)
    else:
        ic_logo_mpl(pwm, ax, f"filter {f}  (consensus {cons}, weight {fc_w[f]:+.2f})")
plt.tight_layout(); plt.savefig("motifs.png", dpi=130, bbox_inches="tight"); plt.show()
for f in top_filters: print(f"filter {f}: consensus {consensus(pwms[int(f)])}")

## 7. Save results

Bundles the metrics, the test-set predictions and the learned PWMs into `results_dna.zip` to download
and commit to the repo.

In [ ]:
import json, zipfile, os
os.makedirs("out", exist_ok=True)
with open("out/metrics.json","w") as fh:
    json.dump({"source": SOURCE, "test_roc_auc": float(test_auc),
               "test_accuracy": float(test_acc), "seq_len": int(LEN),
               "n_train": len(train_seqs), "n_test": len(test_seqs)}, fh, indent=2)
np.savetxt("out/test_predictions.csv",
           np.c_[test_y, p_test], delimiter=",", header="true_label,pred_prob", comments="")
for f, pwm in pwms.items():
    np.savetxt(f"out/pwm_filter{f}.csv", pwm, delimiter=",",
               header="A,C,G,T columns=positions", comments="")
with zipfile.ZipFile("results_dna.zip","w") as z:
    for p in ["out/metrics.json","out/test_predictions.csv","performance.png","motifs.png"]:
        if os.path.exists(p): z.write(p)
    for f in pwms: z.write(f"out/pwm_filter{f}.csv")
from google.colab import files
files.download("results_dna.zip")
print("saved results_dna.zip")